<a href="https://colab.research.google.com/github/switlanakostyuk-ctrl/Apollo/blob/main/%22HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import statsmodels.stats.api as sms
from math import ceil

effect_size = sms.proportion_effectsize(0.20, 0.19)

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

required_n = ceil(required_n)
print(required_n)

24638


Базове утримання 19%


очікуваний ефект 1%.  - тобто треба збільшити до 20%,

рівень значущості α = 0.05 та потужності 0.8.
Отже необхідний розмір вибірки становить 24638 користувачів у кожній групі.

ВИСНОВОК: Потрібно приблизно 24638 користувачів на групу.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [2]:
import pandas as pd

df = pd.read_csv('cookie_cats.csv')

df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


Дані показали що версія з gate_30 демонструє вище утримання користувачів ніж з версією gate_40.

Гіпотеза: можна припустити, що переміщення воріт на 40 рівень погіршує утримання, але це потрібно перевірити статистично.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [3]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

n_con = control.count()
n_treat = treatment.count()

successes = [control.sum(), treatment.sum()]
nobs = [n_con, n_treat]

z_stat, pval = proportions_ztest(successes, nobs=nobs)

(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(
    successes,
    nobs=nobs,
    alpha=0.05 )

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.5f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.00155
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


p-value < 0.05   -   нульова гіпотеза відхиляється

Оскільки різниця між групами є статистично значущою
утримання користувачів дійсно відрізняється між версіями гри.


Довірчі інтервали не перетинаються. Це каже про те що різниця між групами є стабільною. Версія gate_30 статистично краща.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [4]:
import scipy.stats as stats

contingency_table = pd.crosstab(df['version'], df['retention_7'])

chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f'p-value: {p:.5f}')

p-value: 0.00160


отримане p-value 0.00160 є набагато меншим за встановлений "поріг" 0.05, тому нульова гіпотеза відхиляється.

Це каже про те, що зміна версії гри має статистично значущий вплив на те, чи залишаються гравці на 7-й день. Тобто різниця в показниках утримання між групами не є випадковою.

Зміна негативно вплинула на користувачів, і її не варто впроваджувати.